# 01b — Types of Learning and Data Fundamentals

This notebook demonstrates the three types of learning and the core vocabulary of data — pure Python, no libraries.

- Supervised learning: labeled data, predict a target
- Unsupervised learning: no labels, find structure
- Data fundamentals: samples, features, labels, variable types

---
## Part 1: Supervised Learning

We have inputs and correct labels. The model learns to map inputs to labels.

**Task:** Classify emails as spam or not spam based on word counts.

In [1]:
# Supervised learning dataset
# Each sample: (free_count, click_count, label)
# label: 1 = spam, 0 = not spam

emails = [
    # (word 'free' count, word 'click' count, label)
    (3, 2, 1),   # spam
    (5, 4, 1),   # spam
    (4, 1, 1),   # spam
    (0, 0, 0),   # not spam
    (1, 0, 0),   # not spam
    (0, 1, 0),   # not spam
    (2, 0, 0),   # not spam
]

print('Supervised Learning Dataset — Email Spam')
print('-' * 45)
print(f'  {"free":>6}  {"click":>6}  {"label":>8}')
for free, click, label in emails:
    tag = 'spam' if label == 1 else 'not spam'
    print(f'  {free:>6}  {click:>6}  {tag:>8}')

print()
print('Every sample has a label — this is supervised learning.')
print('The model learns: high free/click counts → spam.')

Supervised Learning Dataset — Email Spam
---------------------------------------------
    free   click     label
       3       2      spam
       5       4      spam
       4       1      spam
       0       0  not spam
       1       0  not spam
       0       1  not spam
       2       0  not spam

Every sample has a label — this is supervised learning.
The model learns: high free/click counts → spam.


In [2]:
# Simple rule learned from data: if free + click >= threshold → spam
# Find the best threshold by trying all values

best_threshold = 0
best_accuracy  = 0

for threshold in range(0, 10):
    correct = 0
    for free, click, label in emails:
        pred = 1 if (free + click) >= threshold else 0
        if pred == label:
            correct += 1
    accuracy = correct / len(emails)
    if accuracy > best_accuracy:
        best_accuracy  = accuracy
        best_threshold = threshold

print(f'Best threshold found: free + click >= {best_threshold}')
print(f'Accuracy: {best_accuracy*100:.0f}%')
print()
print('The model searched for the rule — we did not write it.')

Best threshold found: free + click >= 3
Accuracy: 100%

The model searched for the rule — we did not write it.


---
## Part 2: Unsupervised Learning

No labels. The model finds structure on its own.

**Task:** Group customers by purchase behaviour — we do not know in advance how many groups there are or what they represent.

In [3]:
# Unsupervised dataset — no labels
# Each sample: (purchases_per_month, avg_spend)

customers = [
    (1, 20),
    (2, 25),
    (1, 15),
    (2, 30),
    (10, 200),
    (12, 180),
    (11, 220),
    (9,  190),
]

print('Unsupervised Dataset — Customer Behaviour')
print('-' * 40)
print(f'  {"purchases/mo":>12}  {"avg spend":>10}')
for purchases, spend in customers:
    print(f'  {purchases:>12}  {spend:>10}')

print()
print('No labels — we do not know the groups in advance.')

Unsupervised Dataset — Customer Behaviour
----------------------------------------
  purchases/mo   avg spend
             1          20
             2          25
             1          15
             2          30
            10         200
            12         180
            11         220
             9         190

No labels — we do not know the groups in advance.


In [4]:
# Simple clustering: assign each customer to the nearest centroid
# Start with 2 centroids (guessed)

import math

def distance(p1, p2):
    return math.sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)

def mean_point(points):
    n = len(points)
    return (sum(p[0] for p in points)/n, sum(p[1] for p in points)/n)

# Initial centroids
centroids = [(2, 25), (10, 200)]

for iteration in range(5):
    # Assign each customer to nearest centroid
    clusters = [[], []]
    for customer in customers:
        d0 = distance(customer, centroids[0])
        d1 = distance(customer, centroids[1])
        clusters[0 if d0 < d1 else 1].append(customer)

    # Update centroids
    new_centroids = [mean_point(c) for c in clusters]

    if new_centroids == centroids:
        print(f'Converged at iteration {iteration+1}')
        break
    centroids = new_centroids

print()
print('Cluster 0 — Low frequency buyers:')
for c in clusters[0]:
    print(f'  purchases={c[0]}, spend={c[1]}')

print()
print('Cluster 1 — High frequency buyers:')
for c in clusters[1]:
    print(f'  purchases={c[0]}, spend={c[1]}')

print()
print('No labels were given — the algorithm discovered the groups.')

Converged at iteration 2

Cluster 0 — Low frequency buyers:
  purchases=1, spend=20
  purchases=2, spend=25
  purchases=1, spend=15
  purchases=2, spend=30

Cluster 1 — High frequency buyers:
  purchases=10, spend=200
  purchases=12, spend=180
  purchases=11, spend=220
  purchases=9, spend=190

No labels were given — the algorithm discovered the groups.


---
## Part 3: Reinforcement Learning — Concept Demo

An agent learns by trial and error — no dataset, no labels.
It receives rewards for good actions and penalties for bad ones.

**Task:** An agent on a 5-position line must reach position 4 (the goal).
It learns which direction to move by receiving rewards.

In [5]:
import random
random.seed(42)

# Environment: positions 0 to 4, goal at position 4
# Actions: 0 = move left, 1 = move right
# Reward: +1 if reach goal, -1 if fall off left edge, 0 otherwise

# Q-table: Q[position][action] = expected reward
Q = [[0.0, 0.0] for _ in range(5)]

lr      = 0.5   # learning rate
gamma   = 0.9   # discount factor
epsilon = 0.3   # exploration rate

def step(pos, action):
    new_pos = pos + (1 if action == 1 else -1)
    if new_pos > 4:
        return 4, +1, True    # reached goal
    elif new_pos < 0:
        return 0, -1, True    # fell off
    return new_pos, 0, False  # still going

# Training
for episode in range(200):
    pos = random.randint(0, 3)  # start anywhere except goal
    for _ in range(20):
        # Epsilon-greedy: explore randomly or exploit best known action
        if random.random() < epsilon:
            action = random.randint(0, 1)
        else:
            action = 0 if Q[pos][0] > Q[pos][1] else 1

        new_pos, reward, done = step(pos, action)

        # Q-learning update
        best_next = max(Q[new_pos])
        Q[pos][action] += lr * (reward + gamma * best_next - Q[pos][action])

        pos = new_pos
        if done:
            break

print('Learned Q-values (expected reward per action at each position)')
print('-' * 55)
print(f'  {"Position":>8}  {"Move Left":>10}  {"Move Right":>10}  {"Best Action":>12}')
for pos in range(5):
    best = 'right' if Q[pos][1] > Q[pos][0] else 'left'
    print(f'  {pos:>8}  {Q[pos][0]:>10.3f}  {Q[pos][1]:>10.3f}  {best:>12}')

print()
print('The agent learned to always move right — no labels, no dataset.')
print('It discovered the strategy purely from rewards and penalties.')

Learned Q-values (expected reward per action at each position)
-------------------------------------------------------
  Position   Move Left  Move Right   Best Action
         0       4.621       6.560         right
         1       5.904       7.289         right
         2       6.560       8.099         right
         3       7.289       8.999         right
         4       8.099       9.999         right

The agent learned to always move right — no labels, no dataset.
It discovered the strategy purely from rewards and penalties.


---
## Part 4: Data Fundamentals

Every ML system is built on data. This section demonstrates the core vocabulary with concrete Python objects.

In [6]:
# A dataset is a list of samples
# Each sample has features and a label

# Dataset: student records
# Features: age (numerical), study_hours (numerical), grade (categorical)
# Label: passed (1) or failed (0)

dataset = [
    # (age, study_hours, grade,  label)
    (22,   8,           'A',    1),
    (25,   5,           'B',    1),
    (21,   2,           'D',    0),
    (23,   6,           'B',    1),
    (24,   1,           'F',    0),
    (22,   3,           'C',    0),
]

print('Dataset Structure')
print('-' * 55)
print(f'  {"Sample":>6}  {"Age":>4}  {"Hours":>6}  {"Grade":>6}  {"Label":>6}')
for i, (age, hours, grade, label) in enumerate(dataset):
    result = 'pass' if label == 1 else 'fail'
    print(f'  {i+1:>6}  {age:>4}  {hours:>6}  {grade:>6}  {result:>6}')

print(f'\nDataset size: {len(dataset)} samples')
print(f'Features per sample: 3 (age, study_hours, grade)')
print(f'Label: pass/fail')

Dataset Structure
-------------------------------------------------------
  Sample   Age   Hours   Grade   Label
       1    22       8       A    pass
       2    25       5       B    pass
       3    21       2       D    fail
       4    23       6       B    pass
       5    24       1       F    fail
       6    22       3       C    fail

Dataset size: 6 samples
Features per sample: 3 (age, study_hours, grade)
Label: pass/fail


In [7]:
# Variable types

print('Variable Types in This Dataset')
print('-' * 50)

ages   = [row[0] for row in dataset]
hours  = [row[1] for row in dataset]
grades = [row[2] for row in dataset]

print(f'\nAge values:         {ages}')
print(f'  Type: Numerical, Discrete (whole numbers)')
print(f'  Range: {min(ages)} to {max(ages)}')

print(f'\nStudy hours values: {hours}')
print(f'  Type: Numerical, Continuous (could be 5.5, 7.3, etc.)')
print(f'  Range: {min(hours)} to {max(hours)}')

print(f'\nGrade values:       {grades}')
print(f'  Type: Categorical, Ordinal (A > B > C > D > F)')
print(f'  Unique values: {sorted(set(grades))}')

print()
print('Why types matter:')
print('  Numerical → can compute mean, distance, gradient')
print('  Categorical → must encode before feeding to a model')
print('  Ordinal → has order but gaps between values may not be equal')

Variable Types in This Dataset
--------------------------------------------------

Age values:         [22, 25, 21, 23, 24, 22]
  Type: Numerical, Discrete (whole numbers)
  Range: 21 to 25

Study hours values: [8, 5, 2, 6, 1, 3]
  Type: Numerical, Continuous (could be 5.5, 7.3, etc.)
  Range: 1 to 8

Grade values:       ['A', 'B', 'D', 'B', 'F', 'C']
  Type: Categorical, Ordinal (A > B > C > D > F)
  Unique values: ['A', 'B', 'C', 'D', 'F']

Why types matter:
  Numerical → can compute mean, distance, gradient
  Categorical → must encode before feeding to a model
  Ordinal → has order but gaps between values may not be equal


In [8]:
# Labeled vs unlabeled data

labeled   = [(row[0], row[1], row[3]) for row in dataset]          # with labels
unlabeled = [(row[0], row[1])         for row in dataset]          # without labels

print('Labeled data (supervised learning):')
for age, hours, label in labeled:
    print(f'  age={age}, hours={hours}  →  label={label}')

print()
print('Unlabeled data (unsupervised learning):')
for age, hours in unlabeled:
    print(f'  age={age}, hours={hours}  →  (no label)')

print()
print('Labeled data is expensive — requires human annotation.')
print('Unlabeled data is cheap and abundant.')
print('This is why unsupervised learning matters at scale.')

Labeled data (supervised learning):
  age=22, hours=8  →  label=1
  age=25, hours=5  →  label=1
  age=21, hours=2  →  label=0
  age=23, hours=6  →  label=1
  age=24, hours=1  →  label=0
  age=22, hours=3  →  label=0

Unlabeled data (unsupervised learning):
  age=22, hours=8  →  (no label)
  age=25, hours=5  →  (no label)
  age=21, hours=2  →  (no label)
  age=23, hours=6  →  (no label)
  age=24, hours=1  →  (no label)
  age=22, hours=3  →  (no label)

Labeled data is expensive — requires human annotation.
Unlabeled data is cheap and abundant.
This is why unsupervised learning matters at scale.


---
## Summary

| Type | Data | Signal | Example |
|---|---|---|---|
| Supervised | Input + Label | Prediction error | Spam detection |
| Unsupervised | Input only | Data structure | Customer clustering |
| Reinforcement | Actions + Rewards | Reward signal | Game playing |

**Data vocabulary:**
- Sample = one row, one observation
- Feature = one column, one measurable property
- Label = the correct output for a sample
- Numerical = numbers with magnitude (can compute mean, distance)
- Categorical = categories with no inherent magnitude (must encode)